In [ ]:
%cd /kaggle/working

REPO_BRANCH = "tgn-full-ce"

!rm -rf temporal-recsys
!git clone --branch {REPO_BRANCH} --single-branch https://github.com/dshhrv/temporal-recsys.git

%cd /kaggle/working/temporal-recsys


In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from src.tgn_model import TGN

CONFIG_PATH = Path("configs/tgn_ml1m.yaml")

# Set this to a Kaggle input checkpoint path to resume, for example:
# RESUME_CHECKPOINT_PATH = "/kaggle/input/tgn-full-ce-epoch-005/tgn_full_ce_epoch_005.pt"
RESUME_CHECKPOINT_PATH = None
FORCE_RESTART = False


def parse_simple_yaml_value(value):
    value = value.strip()

    if value in {"~", "null", "None"}:
        return None

    if value.lower() == "true":
        return True

    if value.lower() == "false":
        return False

    if value.startswith("[") and value.endswith("]"):
        inner = value[1:-1].strip()

        if not inner:
            return []

        return [parse_simple_yaml_value(part.strip()) for part in inner.split(",")]

    try:
        return int(value)
    except ValueError:
        pass

    try:
        return float(value)
    except ValueError:
        return value.strip('"').strip("'")


def load_simple_yaml(path):
    root = {}
    stack = [(-1, root)]

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].rstrip()

        if not line.strip():
            continue

        indent = len(line) - len(line.lstrip(" "))
        key, value = line.strip().split(":", 1)

        while indent <= stack[-1][0]:
            stack.pop()

        parent = stack[-1][1]

        if value.strip() == "":
            child = {}
            parent[key] = child
            stack.append((indent, child))
        else:
            parent[key] = parse_simple_yaml_value(value)

    return root


config = load_simple_yaml(CONFIG_PATH)

if RESUME_CHECKPOINT_PATH is None:
    RESUME_CHECKPOINT_PATH = config.get("resume_checkpoint_path")

SEED = int(config.get("seed", 42))
USE_GPU = bool(config.get("use_gpu", True))
DEVICE = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")

MAX_EPOCHS = int(config["epochs"])
PATIENCE = int(config.get("stopping_step", 5))
VALID_METRIC = config.get("valid_metric", "Recall@10")
TRAIN_BATCH_SIZE = int(config["train_batch_size"])
EVAL_BATCH_SIZE = int(config["eval_batch_size"])
LEARNING_RATE = float(config["learning_rate"])
WEIGHT_DECAY = float(config.get("weight_decay", 0.0))
EMBEDDING_DIM = int(config["embedding_dim"])
MEMORY_DIM = int(config["memory_dim"])
TIME_DIM = int(config["time_dim"])
EDGE_DIM = int(config.get("edge_dim", 1))
NUM_NEIGHBORS = int(config["num_neighbors"])
NUM_ATTENTION_HEADS = int(config["num_attention_heads"])
DROPOUT = float(config.get("dropout", 0.1))
USE_ITEM_BIAS = bool(config.get("use_item_bias", True))
ITEM_CHUNK_SIZE = int(config.get("item_chunk_size", 256))
TIME_FIELD = config.get("TIME_FIELD", "time_days")
EDGE_FEATURE_FIELD = config.get("EDGE_FEATURE_FIELD", "edge_feat_0")
MASK_SEEN_ITEMS = bool(config.get("eval_args", {}).get("mask_seen_items", True))

OUTPUT_DIR = Path(config.get("output_dir", "/kaggle/working/tgn_full_ce_results"))
CHECKPOINT_DIR = Path(config.get("checkpoint_dir", OUTPUT_DIR / "checkpoints"))
CHECKPOINT_PREFIX = config.get("checkpoint_prefix", "tgn_full_ce")
EPOCH_METRICS_PATH = OUTPUT_DIR / config.get("epoch_metrics_name", "tgn_full_ce_epoch_metrics.csv")
TEST_METRICS_PATH = OUTPUT_DIR / config.get("test_metrics_name", "tgn_full_ce_test_metrics.csv")
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / config.get("best_checkpoint_name", "tgn_full_ce_best.pt")
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / config.get("last_checkpoint_name", "tgn_full_ce_last.pt")
RUN_CONFIG_PATH = OUTPUT_DIR / "tgn_full_ce_run_config.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Config: {CONFIG_PATH}")
print(f"Resume checkpoint: {RESUME_CHECKPOINT_PATH}")


In [ ]:
DATA_DIR = Path(config["data_path"])

if not DATA_DIR.exists() and Path("data/tgn_ml1m").exists():
    DATA_DIR = Path("data/tgn_ml1m")

INTERACTIONS_PATH = DATA_DIR / config.get("interactions_path", "interactions.csv")
META_PATH = DATA_DIR / config.get("meta_path", "meta.json")

if not INTERACTIONS_PATH.exists():
    raise FileNotFoundError(f"Interactions file not found: {INTERACTIONS_PATH}")

metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
interactions = pd.read_csv(INTERACTIONS_PATH)

NUM_USERS = int(metadata["n_users"])
NUM_ITEMS = int(metadata["n_items"])

interactions["user_idx"] = interactions["src"].astype(np.int64)
interactions["item_idx"] = (interactions["dst"] - NUM_USERS).astype(np.int64)

if EDGE_FEATURE_FIELD not in interactions.columns:
    if "rating" not in interactions.columns:
        raise RuntimeError(f"Missing {EDGE_FEATURE_FIELD} and rating columns")

    interactions[EDGE_FEATURE_FIELD] = (interactions["rating"] / 5.0).astype("float32")

if not interactions["user_idx"].between(0, NUM_USERS - 1).all():
    raise RuntimeError("user_idx is out of range")

if not interactions["item_idx"].between(0, NUM_ITEMS - 1).all():
    raise RuntimeError("item_idx is out of range")

sort_columns = [TIME_FIELD]

if "event_id" in interactions.columns:
    sort_columns.append("event_id")

interactions = interactions.sort_values(sort_columns, kind="stable").reset_index(drop=True)
TRAIN = interactions[interactions["split"] == "train"].copy().reset_index(drop=True)
VALID = interactions[interactions["split"] == "valid"].copy().reset_index(drop=True)
TEST = interactions[interactions["split"] == "test"].copy().reset_index(drop=True)

warm_item_set = set(TRAIN["item_idx"].unique().tolist())

if not VALID["item_idx"].isin(warm_item_set).all() or not TEST["item_idx"].isin(warm_item_set).all():
    raise RuntimeError("Cold items remain in valid/test. Regenerate data/tgn_ml1m with cold-item filtering.")

catalog_item_ids = torch.arange(NUM_ITEMS, dtype=torch.long, device=DEVICE)

print(f"Data dir: {DATA_DIR}")
print(f"Train: {len(TRAIN):,}")
print(f"Valid: {len(VALID):,}")
print(f"Test: {len(TEST):,}")
print(f"Users: {NUM_USERS:,}")
print(f"Catalog: {NUM_ITEMS:,} items")
print(f"Time field: {TIME_FIELD}")
print(f"Edge feature: {EDGE_FEATURE_FIELD}")


In [ ]:
def make_temporal_batches(frame, batch_size):
    user_ids = torch.tensor(frame["user_idx"].to_numpy(), dtype=torch.long)
    item_ids = torch.tensor(frame["item_idx"].to_numpy(), dtype=torch.long)
    timestamps = torch.tensor(frame[TIME_FIELD].to_numpy(dtype=np.float32), dtype=torch.float32)
    edge_features = torch.tensor(frame[EDGE_FEATURE_FIELD].to_numpy(dtype=np.float32), dtype=torch.float32).unsqueeze(-1)
    times = timestamps.numpy()
    batches = []
    start = 0

    while start < len(frame):
        end = min(start + batch_size, len(frame))

        while end < len(frame) and times[end] == times[end - 1]:
            end += 1

        batches.append((user_ids[start:end], item_ids[start:end], timestamps[start:end], edge_features[start:end]))
        start = end

    return batches

train_loader = make_temporal_batches(TRAIN, TRAIN_BATCH_SIZE)
valid_loader = make_temporal_batches(VALID, EVAL_BATCH_SIZE)
test_loader = make_temporal_batches(TEST, EVAL_BATCH_SIZE)
train_seen = [set() for _ in range(NUM_USERS)]

for user_id, item_id in zip(TRAIN["user_idx"].to_numpy(), TRAIN["item_idx"].to_numpy()):
    train_seen[int(user_id)].add(int(item_id))


def copy_seen(seen_items):
    return [items.copy() for items in seen_items]


def add_frame_to_seen(seen_items, frame):
    for user_id, item_id in zip(frame["user_idx"].to_numpy(), frame["item_idx"].to_numpy()):
        seen_items[int(user_id)].add(int(item_id))


print(f"Train batches: {len(train_loader):,}")
print(f"Valid batches: {len(valid_loader):,}")
print(f"Test batches: {len(test_loader):,}")


In [ ]:
model = TGN(NUM_USERS, NUM_ITEMS, node_dim=EMBEDDING_DIM, memory_dim=MEMORY_DIM, time_dim=TIME_DIM, edge_dim=EDGE_DIM, num_neighbors=NUM_NEIGHBORS, num_heads=NUM_ATTENTION_HEADS, dropout=DROPOUT, use_item_bias=USE_ITEM_BIAS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
best_valid_score = -float("inf")
best_epoch = None
best_valid_metrics = None
best_model_state_dict = None
bad_epochs = 0
start_epoch = 1

run_config = dict(config)
run_config["resolved_data_dir"] = str(DATA_DIR)
run_config["device"] = str(DEVICE)
run_config["num_users"] = NUM_USERS
run_config["num_items"] = NUM_ITEMS
run_config["resume_checkpoint_path"] = RESUME_CHECKPOINT_PATH

with open(RUN_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)


def copy_model_state_dict():
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def checkpoint_path_for_epoch(epoch):
    return CHECKPOINT_DIR / f"{CHECKPOINT_PREFIX}_epoch_{epoch:03d}.pt"


def make_checkpoint(epoch, row, valid_metrics, is_best, model_state_dict):
    return {"epoch": epoch, "model_state_dict": model_state_dict, "optimizer_state_dict": optimizer.state_dict(), "valid_metrics": valid_metrics, "best_valid_metrics": best_valid_metrics, "best_model_state_dict": best_model_state_dict, "best_valid_score": best_valid_score, "best_epoch": best_epoch, "bad_epochs": bad_epochs, "history": history, "config": run_config, "is_best": is_best, "last_epoch_metrics": row, "checkpoint_state": "after_train_before_validation"}


if RESUME_CHECKPOINT_PATH and not FORCE_RESTART:
    resume_path = Path(RESUME_CHECKPOINT_PATH)

    if not resume_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")

    resume_checkpoint = load_checkpoint(resume_path)
    model.load_state_dict(resume_checkpoint["model_state_dict"])

    if "optimizer_state_dict" in resume_checkpoint:
        optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])

    history = resume_checkpoint.get("history", [])
    best_valid_score = float(resume_checkpoint.get("best_valid_score", -float("inf")))
    best_epoch = resume_checkpoint.get("best_epoch", resume_checkpoint.get("epoch"))
    best_valid_metrics = resume_checkpoint.get("best_valid_metrics", resume_checkpoint.get("valid_metrics"))
    best_model_state_dict = resume_checkpoint.get("best_model_state_dict", copy_model_state_dict())
    bad_epochs = int(resume_checkpoint.get("bad_epochs", 0))
    start_epoch = int(resume_checkpoint["epoch"]) + 1
    synthetic_best_checkpoint = {"epoch": best_epoch, "model_state_dict": best_model_state_dict, "valid_metrics": best_valid_metrics, "best_valid_metrics": best_valid_metrics, "best_model_state_dict": best_model_state_dict, "best_valid_score": best_valid_score, "best_epoch": best_epoch, "history": history, "config": run_config, "checkpoint_state": "after_train_before_validation"}
    torch.save(synthetic_best_checkpoint, BEST_CHECKPOINT_PATH)
    print(f"Resumed from epoch {resume_checkpoint['epoch']} -> next epoch {start_epoch}")
else:
    print("Starting from scratch")

print(model)
print(f"Epochs: {MAX_EPOCHS} | start_epoch: {start_epoch}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")
print(f"Last checkpoint: {LAST_CHECKPOINT_PATH}")


In [ ]:
TOPK = [int(k) for k in config.get("topk", [5, 10, 20])]
MAX_K = max(TOPK)


def update_metrics(totals, scores, targets):
    top_items = scores.topk(MAX_K, dim=1).indices
    matches = top_items.eq(targets.unsqueeze(1)).float()
    ranks = torch.arange(1, MAX_K + 1, device=scores.device, dtype=scores.dtype).unsqueeze(0)

    for k in TOPK:
        matches_at_k = matches[:, :k]
        ranks_at_k = ranks[:, :k]
        totals[f"Recall@{k}"] += matches_at_k.any(dim=1).float().sum().item()
        totals[f"NDCG@{k}"] += (matches_at_k / torch.log2(ranks_at_k + 1.0)).sum(dim=1).sum().item()
        totals[f"MRR@{k}"] += (matches_at_k / ranks_at_k).sum(dim=1).sum().item()

    totals["count"] += targets.size(0)


def seen_before_current_events(user_ids, item_ids, timestamps, seen_items):
    users = user_ids.detach().cpu().tolist()
    items = item_ids.detach().cpu().tolist()
    times = timestamps.detach().cpu().tolist()
    row_seen = [None] * len(users)
    start = 0

    while start < len(users):
        end = start + 1

        while end < len(users) and times[end] == times[start]:
            end += 1

        for row in range(start, end):
            row_seen[row] = seen_items[users[row]].copy()

        for row in range(start, end):
            seen_items[users[row]].add(items[row])

        start = end

    return row_seen


def mask_unavailable_items(scores, targets, row_seen):
    if not MASK_SEEN_ITEMS:
        return

    target_items = targets.detach().cpu().tolist()

    for row, (target_item_id, seen) in enumerate(zip(target_items, row_seen)):
        blocked = seen - {target_item_id}

        if blocked:
            blocked_ids = torch.tensor(list(blocked), dtype=torch.long, device=DEVICE)
            scores[row, blocked_ids] = -torch.inf


@torch.no_grad()
def evaluate(loader, seen_items):
    model.eval()
    totals = {"count": 0}

    for k in TOPK:
        totals[f"Recall@{k}"] = 0.0
        totals[f"NDCG@{k}"] = 0.0
        totals[f"MRR@{k}"] = 0.0

    for user_ids, item_ids, timestamps, edge_features in tqdm(loader, leave=False):
        user_ids = user_ids.to(DEVICE, non_blocking=True)
        item_ids = item_ids.to(DEVICE, non_blocking=True)
        timestamps = timestamps.to(DEVICE, non_blocking=True)
        edge_features = edge_features.to(DEVICE, non_blocking=True)
        model.flush_pending_messages()
        scores = model(user_ids, timestamps, item_chunk_size=ITEM_CHUNK_SIZE)
        row_seen = seen_before_current_events(user_ids, item_ids, timestamps, seen_items)
        mask_unavailable_items(scores, item_ids, row_seen)
        update_metrics(totals, scores, item_ids)
        model.store_batch(user_ids, item_ids, timestamps, edge_features)

    model.flush_pending_messages()
    model.detach_state()
    return {name: value / totals["count"] for name, value in totals.items() if name != "count"}


@torch.no_grad()
def warm_up(loader):
    model.eval()

    for user_ids, item_ids, timestamps, edge_features in tqdm(loader, leave=False):
        user_ids = user_ids.to(DEVICE, non_blocking=True)
        item_ids = item_ids.to(DEVICE, non_blocking=True)
        timestamps = timestamps.to(DEVICE, non_blocking=True)
        edge_features = edge_features.to(DEVICE, non_blocking=True)
        model.flush_pending_messages()
        model.store_batch(user_ids, item_ids, timestamps, edge_features)

    model.flush_pending_messages()
    model.detach_state()


In [ ]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for user_ids, item_ids, timestamps, edge_features in tqdm(train_loader, leave=False):
        user_ids = user_ids.to(DEVICE, non_blocking=True)
        item_ids = item_ids.to(DEVICE, non_blocking=True)
        timestamps = timestamps.to(DEVICE, non_blocking=True)
        edge_features = edge_features.to(DEVICE, non_blocking=True)
        model.flush_pending_messages()
        optimizer.zero_grad(set_to_none=True)
        scores = model(user_ids, timestamps, item_chunk_size=ITEM_CHUNK_SIZE)
        loss = F.cross_entropy(scores, item_ids)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * item_ids.size(0)
        total_examples += item_ids.size(0)
        model.store_batch(user_ids, item_ids, timestamps, edge_features)
        model.detach_state()

    model.flush_pending_messages()
    model.detach_state()
    return total_loss / total_examples


run_started = time.perf_counter()

if start_epoch <= MAX_EPOCHS:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_started = time.perf_counter()
        model.reset_state()
        train_started = time.perf_counter()
        train_loss = train_one_epoch(epoch)
        train_seconds = time.perf_counter() - train_started
        train_state_dict = copy_model_state_dict()
        validation_started = time.perf_counter()
        valid_metrics = evaluate(valid_loader, copy_seen(train_seen))
        validation_seconds = time.perf_counter() - validation_started
        epoch_seconds = time.perf_counter() - epoch_started
        valid_score = valid_metrics[VALID_METRIC]
        is_best = valid_score > best_valid_score

        if is_best:
            best_valid_score = valid_score
            best_epoch = epoch
            best_valid_metrics = valid_metrics
            best_model_state_dict = train_state_dict
            bad_epochs = 0
        else:
            bad_epochs += 1

        row = {"epoch": epoch, "train_loss": train_loss, **valid_metrics, "is_best": is_best, "best_epoch": best_epoch, "best_valid_score": best_valid_score, "bad_epochs": bad_epochs, "train_seconds": train_seconds, "validation_seconds": validation_seconds, "epoch_seconds": epoch_seconds, "cumulative_seconds": time.perf_counter() - run_started}
        history.append(row)
        history_df = pd.DataFrame(history)
        history_df.to_csv(EPOCH_METRICS_PATH, index=False)
        checkpoint = make_checkpoint(epoch, row, valid_metrics, is_best, train_state_dict)
        epoch_checkpoint_path = checkpoint_path_for_epoch(epoch)
        torch.save(checkpoint, epoch_checkpoint_path)
        torch.save(checkpoint, LAST_CHECKPOINT_PATH)

        if is_best:
            torch.save(checkpoint, BEST_CHECKPOINT_PATH)

        print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss {train_loss:.5f} | Recall@5 {valid_metrics['Recall@5']:.5f} | Recall@10 {valid_metrics['Recall@10']:.5f} | Recall@20 {valid_metrics['Recall@20']:.5f} | best_epoch {best_epoch} | bad_epochs {bad_epochs}")
        print(f"Saved epoch checkpoint: {epoch_checkpoint_path}")

        if bad_epochs >= PATIENCE:
            print(f"Early stopping: {VALID_METRIC} did not improve for {PATIENCE} epoch(s).")
            break
else:
    history_df = pd.DataFrame(history)
    print(f"Checkpoint already reached epoch {start_epoch - 1}; MAX_EPOCHS={MAX_EPOCHS}, no extra training.")

history_df = pd.DataFrame(history)
history_df.to_csv(EPOCH_METRICS_PATH, index=False)
print(f"Training block finished in {(time.perf_counter() - run_started) / 3600:.2f} h")
print(f"Best epoch: {best_epoch} | best {VALID_METRIC}: {best_valid_score:.6f}")
print(f"Best checkpoint saved to: {BEST_CHECKPOINT_PATH}")
print(f"Last checkpoint saved to: {LAST_CHECKPOINT_PATH}")


In [ ]:
if BEST_CHECKPOINT_PATH.exists():
    best_checkpoint = load_checkpoint(BEST_CHECKPOINT_PATH)
elif best_model_state_dict is not None:
    best_checkpoint = {"epoch": best_epoch, "model_state_dict": best_model_state_dict, "valid_metrics": best_valid_metrics, "best_valid_metrics": best_valid_metrics, "best_valid_score": best_valid_score, "best_epoch": best_epoch, "history": history, "config": run_config, "checkpoint_state": "after_train_before_validation"}
else:
    best_checkpoint = load_checkpoint(LAST_CHECKPOINT_PATH)

model.load_state_dict(best_checkpoint["model_state_dict"])
valid_recheck = evaluate(valid_loader, copy_seen(train_seen))
print("\nValidation recheck from best checkpoint")
print("Stored best valid:", best_checkpoint.get("valid_metrics"))
print("Recomputed valid:", valid_recheck)

model.load_state_dict(best_checkpoint["model_state_dict"])
warm_up(valid_loader)

test_seen = copy_seen(train_seen)
add_frame_to_seen(test_seen, VALID)

test_started = time.perf_counter()
test_metrics = evaluate(test_loader, test_seen)
test_seconds = time.perf_counter() - test_started

stored_best_metrics = best_checkpoint.get("valid_metrics") or {}
test_results = pd.DataFrame([{"best_epoch": int(best_checkpoint["epoch"]), f"best_valid_{VALID_METRIC}": float(stored_best_metrics.get(VALID_METRIC, best_valid_score)), **test_metrics, "test_seconds": test_seconds}])
test_results.to_csv(TEST_METRICS_PATH, index=False)

print("\nTest metrics from best validation checkpoint")
print(test_results)
print(f"Test time: {test_seconds / 3600:.2f} h")


In [ ]:
history_df


In [ ]:
test_results


In [ ]:
for path in [EPOCH_METRICS_PATH, TEST_METRICS_PATH, BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH, RUN_CONFIG_PATH]:
    if path.exists():
        print(path)

for path in sorted(CHECKPOINT_DIR.glob(f"{CHECKPOINT_PREFIX}_epoch_*.pt")):
    print(path)
